# Lab: Decision Tree và Random Forest

## 1. Decision Tree là gì?

Decision Tree (cây quyết định) là một mô hình phân loại/hồi quy dạng *cây nhị phân* (hoặc đa nhánh). Ở mỗi nút trong, mô hình hỏi một câu *về* dữ liệu (kiểu "tuổi > 30 không?"); rẽ trái nếu đúng, rẽ phải nếu sai. Cứ thế cho đến khi đến lá, nơi chứa nhãn dự đoán.

Ưu điểm cực lớn: **dễ giải thích**. In cây ra, người không biết Machine Learning vẫn đọc được. Đây là điều mà neural network không làm được.

## 2. Cây học bằng cách nào?

Quy trình chia (split) một nút:
1. Thử mọi feature, mọi ngưỡng có thể.
2. Với mỗi cách chia, đo "chất lượng" bằng một tiêu chí (impurity).
3. Chọn cách chia tốt nhất.
4. Lặp lại đệ quy cho hai nhánh con.
5. Dừng khi: nút thuần (chỉ còn 1 lớp), hoặc đạt `max_depth`, hoặc số mẫu < `min_samples_split`.

## 3. Hai tiêu chí impurity phổ biến

### 3.1. Entropy (information gain, thuật toán ID3/C4.5)

Đo độ "hỗn loạn" của một tập:
$$
H(S) = -\sum_{c=1}^{C} p_c \log_2 p_c
$$
với $p_c$ là tỷ lệ lớp $c$ trong $S$.

**Information Gain** = entropy giảm sau khi chia:
$$
IG(S, \text{split}) = H(S) - \sum_{i} \frac{|S_i|}{|S|} H(S_i)
$$

### 3.2. Gini index (thuật toán CART, sklearn dùng mặc định)
$$
G(S) = 1 - \sum_{c=1}^{C} p_c^2
$$

Cả hai đều đạt cực tiểu (= 0) khi nút thuần (chỉ 1 lớp), cực đại khi các lớp đều nhau. Trong thực tế cho kết quả gần nhau, chỉ khác là Gini tính nhanh hơn (không có log).

### Minh hoạ: ba tiêu chí impurity trông như thế nào

![Entropy, Gini và Misclassification error theo p](images/01_entropy_gini_misclassification.png)

*Trái: cả ba đều bằng 0 ở hai đầu ($p=0$ hoặc $p=1$: nút THUẦN, không cần chia nữa) và đạt cực đại tại $p=0.5$ (nút hỗn loạn nhất). Phải: chia Entropy cho 2 để đưa về cùng đỉnh với Gini: hai đường gần như trùng nhau, đó là lý do đổi `criterion='gini'` sang `'entropy'` hầu như không đổi kết quả.*

Ba nhận xét quan trọng mà hình này trả lời:

1. **Vì sao dùng entropy hay gini đều được?** Vì trên đoạn $[0,1]$ chúng chỉ lệch nhau vài phần trăm (vùng cam). Gini rẻ hơn vì không có `log`, và đó là toàn bộ lý do sklearn chọn Gini làm mặc định.
2. **Vì sao KHÔNG dùng misclassification error để mọc cây?** Vì nó chỉ **tuyến tính từng khúc**, không lõm chặt. Hệ quả: có những phép chia rất hữu ích mà gain của nó vẫn bằng đúng 0 → cây dừng sớm oan. Entropy và Gini lõm **chặt** nên mọi phép chia không tầm thường đều cho gain dương.
3. **Ví dụ cụ thể của bẫy trên**: nút cha có (400 lớp 0, 400 lớp 1). Phép chia A cho hai con (300, 100) và (100, 300); phép chia B cho (200, 400) và (200, 0). Cả hai đều có misclassification gain bằng đúng $0.25$ nên không phân biệt được, dù chia B tạo ra một nút **thuần** còn chia A thì không. Gini và entropy thì chấm điểm B cao hơn hẳn.

Với bài toán $C$ lớp, công thức tổng quát là:

$$
H(S) = -\sum_{c=1}^{C} p_c \log_2 p_c \in [0, \log_2 C], \qquad
G(S) = 1 - \sum_{c=1}^{C} p_c^2 \in \left[0, 1 - \tfrac{1}{C}\right]
$$


### Minh hoạ: cây làm gì ở ĐÚNG MỘT nút

![Một bước split: quét ngưỡng, tính Information Gain, chọn ngưỡng tốt nhất](images/02_mot_buoc_split.png)

*Trái: với một feature liên tục, ứng viên ngưỡng là **trung điểm giữa hai giá trị liền kề** đã sắp xếp, nên có $N-1$ ứng viên chứ không phải vô hạn. Giữa: chấm điểm từng ngưỡng bằng Information Gain (và Gini gain): cả hai đường đạt cực đại ở gần như cùng một chỗ. Phải: cây con thu được sau bước chia đó, kèm phép tính IG đầy đủ.*

Điểm mấu chốt để không hiểu sai: cây **không** thử "mọi số thực". Nó sắp xếp các giá trị của feature rồi chỉ thử $N-1$ trung điểm, vì mọi ngưỡng nằm giữa hai giá trị liền kề đều cho ra **cùng một phép chia**.


### Minh hoạ: vì sao cây "leo bậc thang"

![Ranh giới song song trục so với ranh giới chéo](images/03_ranh_gioi_bac_thang.png)

*Cùng độ khó về mặt trực giác, nhưng: ranh giới **thẳng đứng** chỉ tốn 2 lá và sâu 1; ranh giới **chéo 45°** tốn 17 lá và sâu 7 mà vẫn chỉ là xấp xỉ răng cưa. Logistic Regression giải bài toán thứ hai bằng đúng MỘT đường thẳng.*

Lý do nằm ở dạng câu hỏi mà cây được phép hỏi: mỗi nút chỉ hỏi được `x_j <= t?`, tức là một siêu phẳng **song song trục**. Cây không thể hỏi `x_1 - x_2 <= 0?` trừ khi bạn tự tạo sẵn feature đó.

Hệ quả thực tế:

- Cây **bất biến với mọi phép biến đổi đơn điệu từng feature** (log, căn, chuẩn hoá...). Vì thứ tự các giá trị không đổi thì tập ngưỡng ứng viên không đổi. → **Không cần scale cho cây** (khác hẳn KNN).
- Nhưng cây **rất nhạy với phép XOAY** dữ liệu. Xoay 45° một bài toán tuyến tính dễ là biến nó thành ác mộng cho cây.
- Cách chữa: (a) tạo feature tổ hợp thủ công (`x1 - x2`, tỷ số...); (b) dùng Random Forest / Gradient Boosting để nhiều bậc thang trung bình hoá thành đường mượt; (c) dùng `ExtraTrees` hoặc mô hình tuyến tính nếu ranh giới thật sự chéo.


## 4. Vấn đề lớn: Overfitting

Cây *không bị giới hạn* sẽ học tới khi mỗi lá chỉ có 1 mẫu: train accuracy = 100%, test accuracy thì kém. Cách kiểm soát:

- `max_depth`: giới hạn độ sâu cây.
- `min_samples_split`: số mẫu tối thiểu để tiếp tục chia.
- `min_samples_leaf`: số mẫu tối thiểu trong một lá.
- **Pruning** (cắt tỉa): xây cây đầy đủ rồi cắt nhánh nào không cải thiện validation.

## 5. Random Forest

Một cây dễ overfit. Ý tưởng Random Forest: train **nhiều cây** trên **dữ liệu hơi khác nhau**, rồi bầu chọn.

Hai nguồn ngẫu nhiên:
1. **Bagging (Bootstrap Aggregation)**: mỗi cây được train trên một tập bootstrap (sample có hoàn lại từ tập gốc, cùng cỡ).
2. **Random subspace**: ở **mỗi nút**, chỉ xét một tập con ngẫu nhiên các feature (thường $\sqrt{d}$ feature trong $d$).

Hai cơ chế này làm các cây ít tương quan với nhau → khi bầu chọn, lỗi triệt tiêu nhau. Đây là lý do Random Forest gần như **luôn** tốt hơn Decision Tree đơn lẻ.

Một bonus đẹp: bootstrap loại ra ~37% mẫu mỗi cây (gọi là **out-of-bag**). Có thể dùng OOB làm validation set miễn phí.

### Minh hoạ: `max_depth` và quá trình overfit

![Decision boundary theo max_depth và đường train/test accuracy](images/04_anh_huong_max_depth.png)

*Bốn panel đầu: cùng dữ liệu, chỉ đổi `max_depth`. `max_depth=1` (decision stump) chỉ cắt được một nhát → underfit. `max_depth=None` mọc tới 40 lá, khoanh riêng từng điểm nhiễu thành một hộp, train đạt 100% nhưng test tụt. Panel cuối: đường train leo mãi lên 100% trong khi test quay đầu, chữ ký kinh điển của overfitting; vùng xám giữa hai đường là "mức overfit".*

Bảng các nút vặn kiểm soát độ phức tạp trong sklearn:

| Tham số | Ý nghĩa | Hiệu ứng khi tăng |
|---|---|---|
| `max_depth` | Số tầng tối đa | Tăng → cây phức tạp hơn, dễ overfit |
| `min_samples_split` | Số mẫu tối thiểu để được phép chia tiếp | Tăng → cây đơn giản hơn |
| `min_samples_leaf` | Số mẫu tối thiểu trong mỗi lá | Tăng → cây đơn giản hơn, dự đoán ổn định hơn |
| `max_leaf_nodes` | Số lá tối đa (mọc theo kiểu best-first) | Giới hạn trực tiếp kích thước cây |
| `min_impurity_decrease` | Chỉ chia nếu impurity giảm ít nhất bằng ngưỡng này | Tăng → cắt sớm các phép chia vụn vặt |
| `ccp_alpha` | Cắt tỉa sau khi mọc xong (mục 9) | Tăng → cây bị cắt nhiều hơn |

> Trong thực tế, `min_samples_leaf` (đặt 5 đến 20 với dữ liệu vài nghìn dòng) thường là nút vặn **hiệu quả và an toàn nhất**, vì nó chống trực tiếp cái bệnh "một lá một mẫu".


### Ảnh động: cây sâu dần và khoảnh khắc bắt đầu overfit

![Ranh giới bậc thang và đường accuracy khi max_depth chạy từ 1 lên 10](images/anim_cay_sau_dan.gif)

*Bên trái là ranh giới bậc thang của cây khi `max_depth` chạy từ 1 lên 10, kèm số lá ở mỗi mức. Bên phải là hai đường accuracy được vẽ dần theo từng độ sâu. Hãy chú ý thời điểm hai đường tách nhau: train tiếp tục leo lên gần 98% còn test đạt đỉnh rất sớm rồi đi xuống. Khoảng trắng mở rộng giữa hai đường chính là mức overfit đang lớn dần, và đó là lý do phải giới hạn độ sâu thay vì để cây mọc thoải mái.*


### Minh hoạ: rừng làm mượt biên quyết định

![1 cây, 5 cây, 25 cây, 200 cây](images/07_random_forest_lam_muot.png)

*Màu đậm = rừng rất chắc chắn (xác suất gần 0 hoặc 1); màu nhạt = rừng đang phân vân (xác suất gần 0.5). Một cây chỉ biết trả lời 0 hoặc 1 nên toàn màu đậm và đầy góc cạnh. Càng nhiều cây, xác suất càng mịn và vùng "lưỡng lự" quanh biên hiện ra rõ. Đó chính là phần variance đã được trung bình hoá đi.*

![Test accuracy và OOB score theo số cây](images/08_test_acc_theo_so_cay.png)

*Test accuracy tăng rất nhanh trong khoảng 20 cây đầu rồi **bão hoà**. Quan trọng nhất: nó **không bao giờ tụt xuống** khi thêm cây. Đây là điểm khác biệt căn bản so với `max_depth`: tăng số cây KHÔNG gây overfit, chỉ tốn thêm thời gian. Đường OOB (xanh lá) bám sát test accuracy mà không tiêu tốn một mẫu dữ liệu nào.*

> **Quy tắc thực dụng**: đừng đưa `n_estimators` vào `GridSearchCV`. Cứ đặt nó lớn nhất mức thời gian cho phép (200 đến 500), rồi dành ngân sách tìm kiếm cho `max_features`, `max_depth`, `min_samples_leaf`, những tham số THẬT SỰ đánh đổi bias-variance.


# THỰC HÀNH: Phân loại thuốc với Decision Tree + Random Forest

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

df = pd.read_csv('data/drug200.csv')
print(df.head())
print(f'\nShape: {df.shape}')
print(f'Drug counts:\n{df["Drug"].value_counts()}')

In [ ]:
# Encode categorical features. BP, Cholesterol có thứ tự (LOW < NORMAL < HIGH)
# nên dùng ordinal encoding hợp lý hơn one-hot.
df_enc = df.copy()
df_enc['Sex']         = df_enc['Sex'].map({'F': 0, 'M': 1})
df_enc['BP']          = df_enc['BP'].map({'LOW': 0, 'NORMAL': 1, 'HIGH': 2})
df_enc['Cholesterol'] = df_enc['Cholesterol'].map({'NORMAL': 0, 'HIGH': 1})

X = df_enc.drop('Drug', axis=1).values
le_y = LabelEncoder()
y = le_y.fit_transform(df_enc['Drug'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
feature_names = df_enc.drop('Drug', axis=1).columns.tolist()
class_names = le_y.classes_.tolist()

print(f'Features: {feature_names}')
print(f'Classes:  {class_names}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### 1. Decision Tree đơn lẻ

In [ ]:
# Train cây không giới hạn, sẽ overfit
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
print(f'Cây không giới hạn:')
print(f'  Train acc: {dt_full.score(X_train, y_train)*100:.2f}%')
print(f'  Test  acc: {dt_full.score(X_test, y_test)*100:.2f}%')
print(f'  Độ sâu: {dt_full.get_depth()}, số lá: {dt_full.get_n_leaves()}')

In [ ]:
# Sweep max_depth để xem ảnh hưởng đến overfit
depths = list(range(1, 11))
train_acc, test_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train))
    test_acc.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(depths, [a*100 for a in train_acc], 'o-', label='Train')
plt.plot(depths, [a*100 for a in test_acc],  's-', label='Test')
plt.xlabel('max_depth'); plt.ylabel('Accuracy (%)')
plt.title('Train vs test accuracy theo max_depth')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

best_d = depths[int(np.argmax(test_acc))]
print(f'max_depth tốt nhất: {best_d}, test acc = {max(test_acc)*100:.2f}%')

In [ ]:
# Vẽ cây với max_depth=4 cho dễ nhìn
dt_small = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_small.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(dt_small, feature_names=feature_names, class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title(f'Decision Tree (max_depth=4)  test_acc = {dt_small.score(X_test, y_test)*100:.2f}%')
plt.show()

### Feature importance

Mỗi feature được Decision Tree gán một mức "quan trọng": bằng tổng giảm impurity ở các nút nó tham gia chia.

In [ ]:
imp = pd.Series(dt_small.feature_importances_, index=feature_names).sort_values()
imp.plot.barh(figsize=(7, 3))
plt.xlabel('Importance'); plt.title('Feature importance của Decision Tree')
plt.tight_layout(); plt.show()

### 2. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5,
                            oob_score=True, random_state=42)
rf.fit(X_train, y_train)

print(f'Random Forest (100 cây, max_depth=5):')
print(f'  Train acc: {rf.score(X_train, y_train)*100:.2f}%')
print(f'  Test  acc: {rf.score(X_test, y_test)*100:.2f}%')
print(f'  OOB acc:   {rf.oob_score_*100:.2f}%   (validation miễn phí từ bootstrap)')

In [ ]:
# So sánh DT vs RF khi sweep n_estimators
ns = [1, 5, 10, 25, 50, 100, 200]
rf_scores = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
    m.fit(X_train, y_train)
    rf_scores.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(ns, [s*100 for s in rf_scores], 'o-', label='Random Forest')
plt.axhline(dt_small.score(X_test, y_test) * 100, color='red',
            linestyle='--', label='Decision Tree (max_depth=4)')
plt.xlabel('Số cây'); plt.ylabel('Test accuracy (%)')
plt.title('Càng nhiều cây thì RF càng ổn, đến một điểm bão hoà')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Confusion matrix cho RF
y_pred = rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names); ax.set_yticklabels(class_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix của Random Forest')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

## 6. Thuật toán CART chi tiết

`DecisionTreeClassifier` / `DecisionTreeRegressor` của sklearn là một bản cài đặt tối ưu của **CART** (Classification And Regression Trees, Breiman và cộng sự, 1984).

### 6.1. Vòng lặp tham lam (greedy)

Tại mỗi nút $t$ chứa tập mẫu $S_t$ ($N_t$ mẫu), thuật toán:

1. Với **mọi** feature $j$ và **mọi** ngưỡng ứng viên $\theta$ (các trung điểm giữa hai giá trị liền kề đã sắp xếp), chia $S_t$ thành
$$S_L = \{x \in S_t : x_j \le \theta\}, \qquad S_R = S_t \setminus S_L$$
2. Chấm điểm phép chia bằng **mức giảm impurity có trọng số**:
$$
\Delta I(j, \theta) \;=\; I(S_t) \;-\; \frac{|S_L|}{N_t} I(S_L) \;-\; \frac{|S_R|}{N_t} I(S_R)
$$
3. Chọn $(j^*, \theta^*) = \arg\max_{j,\theta} \Delta I(j,\theta)$, tạo hai nút con, rồi **lặp lại đệ quy**.
4. Dừng khi nút thuần, hoặc chạm một trong các điều kiện dừng (`max_depth`, `min_samples_leaf`, ...).

Chú ý phần **trọng số** $|S_L|/N_t$: nếu bỏ nó đi, thuật toán sẽ thích những phép chia tách ra một nút con tí hon nhưng thuần, một kiểu chia vô dụng.

> Trong `feature_importances_`, sklearn dùng bản có thêm trọng số toàn cục: $\frac{N_t}{N}\Delta I$, nhờ đó một phép chia ở gốc (nhiều mẫu) đóng góp nhiều hơn một phép chia ở lá sâu.

### 6.2. Chi phí tính toán

Sắp xếp mỗi feature tốn $O(N_t \log N_t)$; quét ngưỡng chỉ tốn $O(N_t)$ nếu ta duy trì bộ đếm lớp khi trượt ngưỡng từ trái sang phải. Vậy:

$$
\text{một nút: } O(d\, N_t \log N_t) \qquad\Longrightarrow\qquad \text{cả cây (cân bằng): } O(d\, N \log^2 N)
$$

Đó là lý do cây rất nhanh so với danh tiếng của nó.

### 6.3. Vì sao không tìm cây TỐI ƯU TOÀN CỤC?

Greedy nghĩa là: chọn nhát cắt tốt nhất **ngay bây giờ**, không quay lui. Nó hoàn toàn có thể bỏ lỡ cây tốt hơn. Ví dụ kinh điển là bài toán **XOR**: không feature đơn lẻ nào cho gain > 0 ở gốc, nhưng chia hai tầng thì tách hoàn hảo.

Vậy sao không duyệt hết? Vì **Hyafil & Rivest (1976) đã chứng minh xây dựng cây quyết định nhị phân tối ưu là bài toán NP-đầy đủ (NP-complete)**. Số cây có thể có bùng nổ tổ hợp: chỉ riêng ở gốc đã có $\approx d(N-1)$ lựa chọn, rồi mỗi nhánh lại nhân lên tiếp.

Hướng nghiên cứu hiện đại (`GOSDT`, `OCT`, dùng quy hoạch nguyên) tìm được cây tối ưu thật, nhưng chỉ khả thi với dữ liệu rất nhỏ và cây rất nông. Trong thực hành, ta chấp nhận greedy rồi bù lại bằng **ensemble** (Random Forest / Boosting).

### 6.4. Vài điều sklearn KHÔNG làm (hay bị hiểu nhầm)

- **Không hỗ trợ feature phân loại dạng chuỗi.** Phải encode trước. sklearn coi mọi cột là số và chỉ hỏi `x_j <= θ`, nên nếu bạn ordinal-encode một biến *không có thứ tự*, cây sẽ tạo ra những ngưỡng vô nghĩa kiểu "màu <= 2.5".
- **Không thực hiện nhánh đa chiều (multiway split).** ID3/C4.5 làm, CART thì luôn nhị phân.
- Từ sklearn 1.3 trở đi, cây **có** hỗ trợ giá trị thiếu (`NaN`): khi chia, mẫu thiếu được đưa cả loạt về nhánh làm giảm impurity nhiều hơn.


## 7. Tính tay một bước split bằng số cụ thể

Lấy đúng nút cha trong hình ở mục "cây làm gì ở một nút": $N = 50$ mẫu, trong đó **24 mẫu lớp 0** và **26 mẫu lớp 1**.

**Bước 1: impurity của nút cha.** $p_1 = 26/50 = 0.52$, $p_0 = 0.48$:

$$
H(\text{cha}) = -0.48\log_2 0.48 - 0.52\log_2 0.52 = 0.48(1.0589) + 0.52(0.9434) = \mathbf{0.9989}
$$
$$
G(\text{cha}) = 1 - (0.48^2 + 0.52^2) = 1 - (0.2304 + 0.2704) = \mathbf{0.4992}
$$

**Bước 2: thử ngưỡng $x < 5.16$.** Kết quả:

| Nút | $N$ | lớp 0 | lớp 1 | $p_1$ | Entropy | Gini |
|---|---|---|---|---|---|---|
| Cha | 50 | 24 | 26 | 0.52 | 0.9989 | 0.4992 |
| Con trái ($x < 5.16$) | 25 | 24 | 1 | 0.04 | 0.2423 | 0.0768 |
| Con phải ($x \ge 5.16$) | 25 | 0 | 25 | 1.00 | 0.0000 | 0.0000 |

Kiểm tra con trái: $H = -0.04\log_2 0.04 - 0.96\log_2 0.96 = 0.04(4.6439) + 0.96(0.0589) = 0.2423$. Con phải thuần nên impurity $= 0$.

**Bước 3: mức giảm impurity có trọng số.**

$$
IG = 0.9989 - \frac{25}{50}(0.2423) - \frac{25}{50}(0.0000) = 0.9989 - 0.1211 = \mathbf{0.8777}
$$

$$
\Delta G = 0.4992 - \frac{25}{50}(0.0768) - \frac{25}{50}(0.0000) = 0.4992 - 0.0384 = \mathbf{0.4608}
$$

**Bước 4: so sánh.** Lặp lại phép tính này cho cả 49 ngưỡng ứng viên (và cho *mọi* feature nếu có nhiều feature) rồi lấy giá trị lớn nhất. Ở đây $x < 5.16$ thắng, nên cây đặt câu hỏi đó vào nút gốc.

> **Tự kiểm tra**: hai tiêu chí cho hai con số rất khác nhau ($0.878$ vs $0.461$) nhưng **cùng xếp hạng** các ngưỡng gần như y hệt, đúng như hình so sánh Entropy/Gini ở trên. Vì thế đừng bao giờ so sánh trực tiếp giá trị IG với giá trị Gini gain; chỉ so sánh trong cùng một tiêu chí.


## 8. Cây cho HỒI QUY với `DecisionTreeRegressor`

Đổi thước đo impurity là cây làm được hồi quy. Với `criterion='squared_error'`, "độ vẩn đục" của một nút chính là **phương sai** của $y$ trong nút đó:

$$
I(S_t) \;=\; \frac{1}{N_t}\sum_{i \in S_t} (y_i - \bar{y}_t)^2, \qquad \bar{y}_t = \frac{1}{N_t}\sum_{i \in S_t} y_i
$$

Cực đại hoá mức giảm impurity ở đây tương đương **cực tiểu hoá tổng bình phương sai số** của hai nút con:

$$
\arg\min_{j,\theta} \ \Big[ \sum_{i \in S_L}(y_i - \bar{y}_L)^2 + \sum_{i \in S_R}(y_i - \bar{y}_R)^2 \Big]
$$

**Dự đoán tại một lá là một hằng số**: trung bình của $y$ trong lá (với `squared_error`) hoặc trung vị (với `absolute_error`). Vì thế hàm dự đoán của cây hồi quy luôn là **hàm bậc thang**, giống KNN hồi quy, và cũng **không ngoại suy được** ra ngoài miền dữ liệu train.

| `criterion` | Impurity | Dự đoán ở lá | Ghi chú |
|---|---|---|---|
| `'squared_error'` | phương sai | trung bình | Mặc định, nhanh nhất |
| `'friedman_mse'` | biến thể của MSE có thêm hiệu chỉnh | trung bình | Dùng trong Gradient Boosting, chọn split tốt hơn chút |
| `'absolute_error'` | sai số tuyệt đối trung bình | **trung vị** | Chịu outlier tốt, nhưng **chậm hơn nhiều** |
| `'poisson'` | độ lệch Poisson | trung bình | Cho dữ liệu đếm ($y \ge 0$, nguyên) |

Ba điều dễ sai:

1. **$R^2$ trên train của cây không giới hạn luôn bằng 1.0**, vô nghĩa để đánh giá, đúng như accuracy 100% của cây phân loại.
2. **Không cần chuẩn hoá $X$** (cây bất biến với biến đổi đơn điệu), nhưng **outlier trong $y$** thì ảnh hưởng mạnh vì impurity là phương sai → cân nhắc `absolute_error` hoặc log-transform $y$.
3. Cây hồi quy đơn lẻ rất giật cục. Thực tế người ta hầu như luôn dùng `RandomForestRegressor` hoặc `HistGradientBoostingRegressor`.


In [ ]:
# Demo độc lập: cây cho HỒI QUY, dự đoán là hằng số trên mỗi lá
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 10, 120)).reshape(-1, 1)
y = np.sin(x).ravel() + 0.12 * x.ravel() + rng.normal(0, 0.22, 120)
xs = np.linspace(-1, 11, 500).reshape(-1, 1)     # cố tình vượt ra ngoài miền train

plt.figure(figsize=(9.5, 4.2))
plt.scatter(x, y, s=18, color='k', alpha=.55, label='train')
for d, c in [(1, '#d97706'), (3, '#2563eb'), (6, '#059669'), (None, '#dc2626')]:
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(x, y)
    plt.plot(xs, m.predict(xs), lw=1.8, color=c,
             label=f'max_depth={d} ({m.get_n_leaves()} lá, train R2={m.score(x, y):.3f})')
plt.axvspan(-1, 0, color='gray', alpha=.15)
plt.axvspan(10, 11, color='gray', alpha=.15)
plt.title('Cây hồi quy = hàm BẬC THANG; vùng xám ngoài miền train → chỉ kéo ngang')
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print('max_depth=None cho train R2 = 1.000 → con số này KHÔNG dùng để đánh giá được.')


## 9. Cắt tỉa (pruning): pre-pruning và cost-complexity pruning

### 9.1. Hai trường phái

| | **Pre-pruning** (cắt trước / early stopping) | **Post-pruning** (cắt sau) |
|---|---|---|
| Làm gì | Dừng chia sớm bằng `max_depth`, `min_samples_leaf`, `min_impurity_decrease` | Mọc cây đầy đủ rồi cắt bớt nhánh |
| Ưu | Nhanh, tốn ít bộ nhớ | Nhìn được toàn cảnh trước khi quyết định cắt |
| Nhược | **Thiển cận**: một phép chia hiện tại kém có thể mở đường cho phép chia cực tốt ở tầng dưới (bài toán XOR) | Tốn công mọc cây đầy đủ trước |
| Trong sklearn | các tham số `max_*`, `min_*` | `ccp_alpha` |

### 9.2. Cost-complexity pruning (CCP)

Ý tưởng: đừng chỉ nhìn sai số, hãy **trả tiền cho mỗi chiếc lá**. Với cây con $T$:

$$
R_\alpha(T) \;=\; \underbrace{R(T)}_{\text{sai số / impurity của các lá}} \;+\; \alpha \underbrace{|T|}_{\text{số lá}}
$$

- $\alpha = 0$ → không phạt → cây đầy đủ thắng.
- $\alpha$ rất lớn → mỗi chiếc lá quá đắt → cây co lại còn đúng nút gốc.
- $\alpha$ ở giữa → cây vừa phải. Đây chính xác là dạng regularization giống $\alpha$ của Ridge/Lasso ở bài Hồi quy tuyến tính.

**Thuật toán weakest-link**: với mỗi nút trong $t$, tính

$$
\alpha_{\text{eff}}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}
$$

($R(t)$ = sai số nếu biến $t$ thành lá; $T_t$ = cây con gốc tại $t$). Nút có $\alpha_{\text{eff}}$ **nhỏ nhất** là "mắt xích yếu nhất", cắt nó trước. Lặp lại, ta được một **dãy cây lồng nhau hữu hạn** ứng với một dãy $\alpha$ tăng dần. `clf.cost_complexity_pruning_path(X, y)` trả về đúng dãy $\alpha$ đó.

![Accuracy và số lá theo ccp_alpha](images/05_cost_complexity_pruning.png)

*Trái: cây đầy đủ đạt train 100% nhưng test thấp; tăng $\alpha$ vừa đủ làm **test accuracy TĂNG** trong khi cây co từ 16 lá xuống 7 lá. Phải: số lá và độ sâu giảm **đơn điệu** theo $\alpha$, mỗi bậc thang là một nhánh bị cắt.*

Quy trình chuẩn:

```python
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]              # bỏ alpha cuối (cây chỉ còn gốc)
gs = GridSearchCV(DecisionTreeClassifier(random_state=0),
                  {'ccp_alpha': alphas}, cv=5).fit(X_train, y_train)
best_tree = gs.best_estimator_
```

> Chọn $\alpha$ bằng **cross-validation trên tập train**, không bằng test. Hình trên vẽ theo test chỉ để *minh hoạ* hình dạng đường cong.


## 10. ID3, C4.5 và CART: ba thế hệ cây quyết định

| | **ID3** (Quinlan 1986) | **C4.5** (Quinlan 1993) | **CART** (Breiman 1984) |
|---|---|---|---|
| Tiêu chí chia | Information Gain | **Gain Ratio** | Gini (phân loại) / MSE (hồi quy) |
| Kiểu nhánh | Đa nhánh, mỗi giá trị một nhánh | Đa nhánh (rời rạc), nhị phân (liên tục) | **Luôn nhị phân** |
| Feature liên tục | Không hỗ trợ | Có (tìm ngưỡng tốt nhất) | Có |
| Giá trị thiếu | Không | Có (chia mẫu theo trọng số xuống các nhánh) | Có (surrogate split) |
| Cắt tỉa | Không | Error-based pruning | **Cost-complexity pruning** |
| Hồi quy | Không | Không | **Có** |
| Trong sklearn | không | không | **có** (bản tối ưu) |

### Vì sao C4.5 phải phát minh ra Gain Ratio?

Information Gain có một khuyết tật chí mạng: **nó thiên vị feature có nhiều giá trị**. Cực đoan nhất, hãy tưởng tượng bạn đưa cột `MaKhachHang` (mỗi dòng một giá trị duy nhất) vào ID3. Chia theo cột đó cho ra $N$ nút con, mỗi nút đúng 1 mẫu → mọi nút đều thuần → $IG = H(S)$, giá trị **lớn nhất có thể**. ID3 sẽ chọn nó ngay lập tức và tạo ra một cây hoàn toàn vô dụng.

C4.5 chữa bằng cách chia cho **Split Information**, chính là entropy của *cách chia* (không phải của nhãn):

$$
\text{SplitInfo}(S, A) = -\sum_{i} \frac{|S_i|}{|S|} \log_2 \frac{|S_i|}{|S|},
\qquad
\text{GainRatio}(S, A) = \frac{IG(S, A)}{\text{SplitInfo}(S, A)}
$$

Với cột `MaKhachHang`, $\text{SplitInfo} = \log_2 N$, một giá trị rất lớn, nên Gain Ratio bị kéo tụt xuống. Đó là một cách phạt feature "chia quá vụn".

> CART tránh vấn đề này theo cách khác: **luôn chia nhị phân**, nên một feature 100 mức cũng chỉ tạo ra 2 nhánh. Tuy vậy CART **vẫn còn** một dạng thiên vị nhẹ với feature nhiều giá trị, và nó lộ ra rõ nhất ở `feature_importances_` (xem mục 14).


## 11. Vì sao bagging giảm variance: công thức then chốt

Đây là công thức quan trọng nhất của cả bài học về ensemble. Giả sử ta có $B$ mô hình, mỗi mô hình có phương sai $\sigma^2$, và **mọi cặp** có hệ số tương quan $\rho$. Phương sai của trung bình cộng $\bar{f} = \frac{1}{B}\sum_b f_b$ là:

$$
\boxed{\ \mathrm{Var}(\bar{f}) \;=\; \rho\,\sigma^2 \;+\; \frac{1-\rho}{B}\,\sigma^2\ }
$$

*Chứng minh nhanh*: $\mathrm{Var}(\bar f) = \frac{1}{B^2}\big[\sum_b \mathrm{Var}(f_b) + \sum_{b \ne b'} \mathrm{Cov}(f_b, f_{b'})\big] = \frac{1}{B^2}\big[B\sigma^2 + B(B-1)\rho\sigma^2\big] = \frac{\sigma^2}{B} + \frac{B-1}{B}\rho\sigma^2$, sắp xếp lại ra đúng công thức trên.

Đọc công thức này là hiểu toàn bộ triết lý Random Forest:

- Số hạng thứ hai $\dfrac{1-\rho}{B}\sigma^2$ **tan biến khi $B \to \infty$**. Đó là lý do "thêm cây thì chỉ có lợi rồi bão hoà".
- Số hạng thứ nhất $\rho\sigma^2$ **KHÔNG phụ thuộc $B$**. Nó là **sàn** mà bạn không bao giờ vượt qua được bằng cách thêm cây.

$$
\lim_{B \to \infty} \mathrm{Var}(\bar{f}) = \rho\,\sigma^2
$$

Vậy muốn giỏi hơn, chỉ còn cách **giảm $\rho$**, tức là làm các cây bớt giống nhau. Đó chính xác là lý do tồn tại của hai nguồn ngẫu nhiên trong Random Forest:

| Cơ chế | Giảm cái gì | Chi tiết |
|---|---|---|
| **Bagging** (bootstrap) | $\rho$ | Mỗi cây thấy một tập dữ liệu hơi khác |
| **Random subspace** (`max_features`) | $\rho$ **mạnh hơn nhiều** | Ở MỖI nút chỉ được xét $\sqrt{d}$ feature ngẫu nhiên → hai cây khó cùng chọn một feature thống trị |

Nhưng có đánh đổi: giảm `max_features` làm $\rho$ giảm **nhưng $\sigma^2$ của từng cây lại tăng** (mỗi cây yếu đi vì bị giấu bớt feature). Tích $\rho\sigma^2$ có một điểm cực tiểu ở giữa → đó là lý do `max_features` là siêu tham số đáng tune nhất của Random Forest.

### Hai hệ quả thường bị bỏ qua

1. **Bagging KHÔNG giảm bias.** Trung bình của $B$ mô hình cùng bị lệch một kiểu thì vẫn lệch y như thế. Vì vậy cây cơ sở trong Random Forest phải **sâu** (bias thấp, variance cao), chính là loại cây mà một mình nó thì overfit. Đặt `max_depth=2` cho Random Forest là tự tay vô hiệu hoá nó.
2. **Boosting đi hướng ngược lại**: dùng cây **nông** (bias cao) và giảm bias dần qua từng vòng. Xem mục 15.


In [ ]:
# Demo độc lập: kiểm chứng công thức Var(trung bình) = rho*sigma^2 + (1-rho)/B * sigma^2
import numpy as np

rng = np.random.default_rng(0)
sigma2 = 1.0
print(f'{"rho":>5} | {"B":>4} | {"Var mô phỏng":>13} | {"Var lý thuyết":>14} | {"sàn = rho*sigma2":>17}')
print('-' * 66)
for rho in [0.0, 0.3, 0.8]:
    for B in [1, 5, 50, 1000]:
        # sinh B biến có phương sai 1 và tương quan đôi một = rho
        common = rng.normal(size=(60000, 1)) * np.sqrt(rho)
        indiv = rng.normal(size=(60000, B)) * np.sqrt(1 - rho)
        mean_pred = (common + indiv).mean(axis=1)
        theo = rho * sigma2 + (1 - rho) / B * sigma2
        print(f'{rho:>5.1f} | {B:>4} | {mean_pred.var():>13.4f} | {theo:>14.4f} | {rho*sigma2:>17.4f}')

print('\nrho = 0   : thêm cây giảm variance theo đúng 1/B → về 0.')
print('rho = 0.8 : dù có 1000 cây, variance vẫn kẹt ở 0.8 → phải GIẢM rho (max_features).')


## 12. OOB score: chứng minh con số 36.8%

### 12.1. Chứng minh

Bootstrap = bốc $N$ mẫu **có hoàn lại** từ tập train $N$ mẫu. Xét một mẫu cụ thể $i$:

- Xác suất **không** bốc trúng $i$ ở một lần bốc: $1 - \dfrac{1}{N}$.
- $N$ lần bốc độc lập nhau, nên xác suất $i$ **không** xuất hiện lần nào:

$$
P(i \notin \text{bootstrap}) = \left(1 - \frac{1}{N}\right)^{N}
$$

- Cho $N \to \infty$, dùng $\lim_{N\to\infty}(1 - \tfrac{1}{N})^N = e^{-1}$:

$$
\boxed{\ P(i \text{ là OOB}) \;\longrightarrow\; \frac{1}{e} \;\approx\; 0.3679 \;=\; 36.8\%\ }
$$

Hội tụ **rất nhanh**: $N=10 \to 34.9\%$, $N=100 \to 36.6\%$, $N=1000 \to 36.8\%$.

![Bootstrap sampling và tỷ lệ OOB](images/06_bagging_va_oob.png)

*Trái: mỗi cây bốc lại $N$ mẫu có hoàn lại: vài mẫu bị lấy 2 đến 3 lần (ô xanh đậm), vài mẫu không được lấy lần nào (ô viền đỏ đứt = OOB). Phải: tỷ lệ OOB theo lý thuyết và theo mô phỏng, cùng hội tụ về $1/e$.*

### 12.2. Vì sao đó là "validation miễn phí"

Với mẫu $i$, khoảng **36.8% số cây** trong rừng chưa từng nhìn thấy nó. Chỉ cần lấy riêng nhóm cây đó bỏ phiếu cho $i$ → ta có một dự đoán **hoàn toàn out-of-sample** cho $i$, mà không cần cắt ra một tập validation nào. Làm thế cho mọi $i$ rồi tính accuracy → **OOB score**.

```python
rf = RandomForestClassifier(n_estimators=300, oob_score=True,
                            bootstrap=True, random_state=42).fit(X_train, y_train)
print(rf.oob_score_)          # ước lượng gần như không chệch của test accuracy
```

### 12.3. Những cảnh báo cần nhớ

- **Bắt buộc `bootstrap=True`** (mặc định là True; `ExtraTreesClassifier` mặc định **False** → không có OOB).
- **Cần đủ nhiều cây.** Mỗi mẫu chỉ được $\approx 0.368 \times B$ cây bỏ phiếu. Với $B = 10$ thì chỉ ~4 cây → OOB rất nhiễu và **bi quan** hơn thực tế. Từ $B \ge 100$ trở lên OOB mới đáng tin.
- **OOB không thay thế được test set** khi bạn dùng chính OOB để tune siêu tham số. Lúc đó nó trở thành validation set và cũng có nguy cơ overfit theo cách y hệt.
- Với dữ liệu **có cấu trúc thời gian hoặc nhóm** (time series, nhiều dòng cùng một bệnh nhân), OOB **lạc quan giả tạo** vì các dòng liên quan vẫn nằm trong tập bootstrap. Khi đó phải dùng `TimeSeriesSplit` / `GroupKFold`.


## 13. `max_features`: nguồn ngẫu nhiên quan trọng nhất

Ở **mỗi nút** (chứ không phải mỗi cây), Random Forest chỉ được xét một tập con ngẫu nhiên gồm `max_features` feature. Đây là thứ phân biệt Random Forest với "bagging cây thuần tuý".

| Bài toán | Khuyến nghị kinh điển (Breiman) | Mặc định trong sklearn hiện nay |
|---|---|---|
| Phân loại | $\sqrt{d}$ | `max_features='sqrt'` ✅ trùng khuyến nghị |
| Hồi quy | $d/3$ | `max_features=1.0` (**dùng hết** feature) ⚠️ |

> **Bẫy phiên bản**: `RandomForestRegressor` của sklearn hiện đặt `max_features=1.0`, tức là mỗi nút xét *toàn bộ* feature → các cây tương quan mạnh hơn khuyến nghị gốc. Nếu bài toán hồi quy của bạn có nhiều feature, hãy thử `max_features='sqrt'` hoặc `1/3` trong `GridSearchCV`, thường tốt hơn mặc định rõ rệt.

### Ảnh hưởng khi giảm `max_features`

| | `max_features` nhỏ | `max_features` = d (dùng hết) |
|---|---|---|
| Tương quan $\rho$ giữa các cây | **thấp** ✅ | cao ❌ |
| Sức mạnh từng cây ($\sigma^2$) | yếu hơn ❌ | mạnh nhất ✅ |
| Tốc độ train | **nhanh hơn** | chậm |
| Khi có 1 feature thống trị | rất hữu ích, các cây khác được "ép" phải học feature khác | mọi cây đều chia y hệt ở gốc → gần như 1 cây |

Nhớ công thức mục 11: cái ta tối thiểu hoá là $\rho\sigma^2$ chứ không phải riêng $\rho$ hay riêng $\sigma^2$ → luôn có một điểm ngọt ở giữa.

### Họ hàng: Extra Trees (Extremely Randomized Trees)

`ExtraTreesClassifier` thêm một tầng ngẫu nhiên nữa: thay vì tìm **ngưỡng tốt nhất**, nó bốc **ngưỡng ngẫu nhiên** cho mỗi feature ứng viên rồi chọn cái tốt nhất trong số đó.

- $\rho$ giảm mạnh hơn nữa → variance thấp hơn RF.
- Bias tăng nhẹ.
- Train **nhanh hơn hẳn** (không phải sắp xếp để tìm ngưỡng).
- Mặc định `bootstrap=False` → dùng toàn bộ dữ liệu cho mỗi cây, và **không có OOB** trừ khi bật `bootstrap=True`.

Thực tế: cứ thử cả `RandomForest` lẫn `ExtraTrees`, chúng rẻ và thường chênh nhau 1-2%, đôi khi ExtraTrees thắng.


## 14. Feature importance: `feature_importances_` nói dối như thế nào

### 14.1. Impurity-based importance (MDI): cái sklearn trả về mặc định

$$
\text{importance}(j) \;=\; \sum_{t \,:\, \text{nút } t \text{ chia theo } j} \frac{N_t}{N}\,\Delta I(t)
$$

rồi chuẩn hoá để tổng bằng 1. Nghĩa đen của nó là: *"feature này đã giúp giảm bao nhiêu impurity trên toàn bộ cây"*.

**Ba khuyết tật nghiêm trọng:**

1. **Thiên vị feature có nhiều giá trị (high cardinality).** Một feature liên tục hoặc một cột ID có $N$ giá trị cho ra $N-1$ ngưỡng ứng viên, tức là rất nhiều cơ hội để tình cờ tìm ra một nhát cắt "có vẻ tốt" trên tập train. Một feature nhị phân chỉ có 1 ngưỡng duy nhất.
2. **Được tính trên tập TRAIN.** Nó thưởng cho việc *overfit*: một feature nhiễu giúp cây khoanh riêng vài điểm train vẫn được ghi nhận công.
3. **Feature tương quan chia nhau điểm.** Hai cột gần trùng nhau sẽ chia đôi tầm quan trọng, khiến cả hai trông "không quan trọng lắm".

![Impurity importance vs permutation importance](images/09_feature_importance_thien_vi.png)

*Xanh = feature THẬT SỰ hữu ích, đỏ = nhiễu thuần tuý. Trái: `feature_importances_` chấm hai cột nhiễu liên tục (1500 giá trị khác nhau) gần 0.30, ngang ngửa feature hữu ích thật. Phải: `permutation_importance` đo trên tập TEST đẩy chúng về ~0.006, còn feature thật giữ nguyên 0.227.*

### 14.2. Permutation importance: cách đo trung thực

Ý tưởng cực đơn giản: đo điểm số của mô hình, rồi **xáo trộn ngẫu nhiên cột $j$** (giữ nguyên phân phối biên nhưng phá vỡ liên hệ với $y$), đo lại, lấy phần chênh:

$$
\text{PI}(j) \;=\; s_{\text{gốc}} \;-\; \frac{1}{K}\sum_{k=1}^{K} s_{\text{sau khi xáo cột } j}^{(k)}
$$

```python
from sklearn.inspection import permutation_importance
r = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=0)
for i in r.importances_mean.argsort()[::-1]:
    print(f'{feature_names[i]:<15} {r.importances_mean[i]:.4f} +/- {r.importances_std[i]:.4f}')
```

Ưu điểm: đo trên **dữ liệu chưa thấy**, không thiên vị theo cardinality, dùng được cho **mọi** mô hình (kể cả mạng nơ-ron).

**Nhưng nó cũng có giới hạn**: với hai feature tương quan mạnh, xáo một cột không làm mô hình tệ đi (vì nó vẫn còn cột kia) → **cả hai cùng bị chấm điểm thấp**. Cách chữa: gom cụm feature tương quan (`scipy.cluster.hierarchy` trên ma trận Spearman) rồi xáo cả cụm, hoặc dùng **drop-column importance** (train lại khi bỏ hẳn cột: chính xác nhất nhưng đắt nhất).

### 14.3. Câu quan trọng nhất của cả mục này

> Feature importance là **tương quan**, KHÔNG phải **nhân quả**. Nó trả lời câu "mô hình đã dùng cột nào nhiều", không phải câu "cột nào gây ra $y$". Đừng bao giờ viết trong báo cáo rằng "biến X là nguyên nhân của Y vì importance cao".


In [ ]:
# Demo độc lập: impurity importance BỊ THIÊN VỊ, permutation importance thì không
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 1500
x_bin   = rng.integers(0, 2, n)                    # HỮU ÍCH, chỉ 2 giá trị
x_ord   = rng.integers(0, 4, n)                    # hữu ích vừa, 4 giá trị
noise_c = rng.normal(size=n)                       # NHIỄU, ~1500 giá trị
noise_id = rng.permutation(n).astype(float)        # NHIỄU, ID duy nhất mỗi dòng
noise_b = rng.integers(0, 2, n).astype(float)      # NHIỄU, 2 giá trị

logit = 2.6 * (x_bin - .5) + 0.55 * (x_ord - 1.5)
y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X = np.c_[x_bin, x_ord, noise_c, noise_id, noise_b]
names = ['x_bin(hữu ích)', 'x_ord(hữu ích)', 'nhiễu_liên_tục',
         'nhiễu_ID', 'nhiễu_nhị_phân']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.35, random_state=0, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
perm = permutation_importance(rf, X_te, y_te, n_repeats=25, random_state=0, n_jobs=-1)

print(pd.DataFrame({
    'impurity (MDI)': rf.feature_importances_.round(3),
    'permutation (test)': perm.importances_mean.round(3),
}, index=names).sort_values('permutation (test)', ascending=False))

print('\nHai cột nhiễu có NHIỀU giá trị được MDI chấm điểm rất cao,')
print('nhưng permutation importance vạch trần: xáo trộn chúng không làm mô hình tệ đi.')


## 15. Bagging vs Boosting: hai triết lý ensemble

| | **Bagging** (Random Forest) | **Boosting** (AdaBoost, GBM, XGBoost) |
|---|---|---|
| Cách huấn luyện | **Song song**, các cây độc lập | **Tuần tự**, cây sau sửa lỗi cây trước |
| Mục tiêu chính | Giảm **variance** | Giảm **bias** |
| Cây cơ sở | **Sâu**, mạnh, tự nó overfit | **Nông** (stump đến depth 3-6), yếu |
| Trọng số khi gộp | Bằng nhau (bỏ phiếu / trung bình) | Có trọng số theo chất lượng từng cây |
| Thêm cây quá nhiều | An toàn, chỉ bão hoà | **Có thể overfit** → cần `early_stopping` |
| Tham số nhạy cảm nhất | `max_features` | `learning_rate` × `n_estimators` |
| Song song hoá | Dễ (`n_jobs=-1`) | Khó (bản chất tuần tự) |
| Chịu nhiễu nhãn | Tốt | Kém hơn (cứ cố sửa cho bằng được điểm nhiễu) |
| Cần tune | Ít | Nhiều |

### Các thuật toán boosting đáng biết

- **AdaBoost** (1995): sau mỗi vòng, **tăng trọng số** cho các mẫu bị dự đoán sai để cây sau tập trung vào chúng. Tương đương tối thiểu hoá hàm mất mát mũ $e^{-y f(x)}$. Rất nhạy với nhãn nhiễu.
- **Gradient Boosting**: tổng quát hoá thêm một bước, mỗi cây mới được fit vào **gradient âm của hàm mất mát** (với MSE thì đúng bằng phần dư $y - \hat{y}$). Nhờ đó dùng được với mọi hàm mất mát khả vi (log-loss, Huber, quantile...).
- **XGBoost / LightGBM / CatBoost**: Gradient Boosting công nghiệp hoá: thêm regularization $L_1/L_2$ trên giá trị lá, chia theo histogram thay vì sắp xếp, xử lý thiếu dữ liệu và feature phân loại natively, chạy song song trên nhiều lõi/GPU. Đây là các mô hình **thường thắng trên dữ liệu dạng bảng**, hơn cả deep learning.
- Trong sklearn (không cần cài thêm gì): **`HistGradientBoostingClassifier`**, nhanh gần bằng LightGBM, tự xử lý `NaN`, hỗ trợ `categorical_features`, có `early_stopping='auto'`.

### Chọn cái nào?

| Tình huống | Nên chọn |
|---|---|
| Cần kết quả ổn định, ít công tune, chống nhiễu | **Random Forest** |
| Muốn điểm cao nhất trên dữ liệu bảng, chấp nhận tune | **XGBoost / LightGBM / HistGradientBoosting** |
| Dữ liệu rất nhỏ, nhiều nhiễu nhãn | **Random Forest** (boosting sẽ học thuộc nhiễu) |
| Cần train nhanh trên nhiều lõi | **Random Forest / ExtraTrees** (song song hoàn toàn) |
| Cần giải thích được từng luật cho người dùng cuối | **Một Decision Tree đã cắt tỉa** |


## 16. Bảng tổng hợp: Decision Tree vs Random Forest

| Tiêu chí | **Decision Tree** | **Random Forest** |
|---|---|---|
| Độ chính xác | Trung bình | **Cao**, gần như luôn tốt hơn |
| Overfitting | **Rất dễ** nếu không giới hạn | Kháng tốt (nhờ trung bình hoá) |
| Ổn định (đổi `random_state`) | **Kém**, đổi vài dòng dữ liệu là đổi cả cây | **Cao** |
| Giải thích được | ⭐⭐⭐⭐⭐ In cây ra là đọc được | ⭐⭐ Chỉ còn importance / SHAP |
| Tốc độ train | **Rất nhanh** | Chậm hơn $B$ lần (nhưng song song được) |
| Tốc độ predict | **Rất nhanh** ($O(\text{độ sâu})$) | Chậm hơn $B$ lần |
| Kích thước model | Vài KB | Vài MB đến vài trăm MB |
| Cần scale feature | Không | Không |
| Xử lý feature phân loại | Cần encode | Cần encode |
| Có xác suất tin cậy | Kém (lá thường thuần → 0 hoặc 1) | **Tốt** (tỷ lệ phiếu là ước lượng xác suất hợp lý) |
| Siêu tham số cần tune | `max_depth`, `min_samples_leaf`, `ccp_alpha` | `max_features`, `min_samples_leaf` |

### Khi nào MỘT CÂY vẫn tốt hơn cả rừng

Không phải lúc nào "chính xác hơn" cũng là "tốt hơn". Chọn một cây đơn khi:

1. **Bắt buộc giải thích được từng quyết định.** Ngành tín dụng, bảo hiểm, y tế ở nhiều nước yêu cầu tổ chức phải nêu lý do từ chối một hồ sơ. Một cây đã cắt tỉa cho ra chuỗi luật `if ... then ...` mà người không biết ML vẫn đọc được, còn "trung bình của 300 cây" thì không.
2. **Cần bộ luật để chuyên gia kiểm tra và chỉnh tay.** Ví dụ phác đồ sàng lọc y tế: bác sĩ phải soi được từng nhánh và có thể sửa ngưỡng theo hướng dẫn lâm sàng.
3. **Triển khai trên thiết bị siêu nhẹ** (vi điều khiển, thẻ thông minh): một cây có thể biên dịch thành vài chục câu `if`.
4. **Latency cực thấp / lô dự đoán khổng lồ**: predict của cây là $O(\text{độ sâu})$, của rừng là $B$ lần như thế.
5. **Dạy học và khám phá dữ liệu**: vẽ cây ra là thấy ngay feature nào quan trọng và ngưỡng ở đâu.

> Mẹo dung hoà: train Random Forest để lấy độ chính xác cao, rồi train **một cây nông làm mô hình thay thế (surrogate)** trên *dự đoán của rừng* để giải thích. Bạn có cả hai, nhưng phải nói rõ đó là lời giải thích *xấp xỉ*.


## 17. Danh sách bẫy: kiểm tra trước khi nộp bài

1. **Chuẩn hoá feature cho cây rồi tưởng có ích.** Cây bất biến với mọi phép biến đổi đơn điệu từng feature → `StandardScaler` không làm gì cả (không hại, nhưng cũng vô nghĩa). Ngược lại, KNN thì **bắt buộc**.
2. **Ordinal-encode một biến KHÔNG có thứ tự.** `màu: đỏ=0, xanh=1, vàng=2` khiến cây tạo ngưỡng "màu <= 1.5", một nhóm vô nghĩa. Với biến nominal ít mức, hãy one-hot. (Trong lab này `BP: LOW < NORMAL < HIGH` **có** thứ tự nên ordinal là đúng.)
3. **Báo cáo train accuracy của cây không giới hạn.** Nó gần như luôn bằng 100%, một con số vô nghĩa.
4. **Bật `oob_score=True` với quá ít cây.** Dưới ~100 cây, mỗi mẫu chỉ có vài cây bỏ phiếu → OOB nhiễu và bi quan.
5. **Đặt `max_depth` nhỏ cho Random Forest.** Bagging chỉ giảm variance chứ không giảm bias → cây cơ sở phải **sâu**. `max_depth=3` biến rừng thành một tập hợp mô hình cùng bị lệch.
6. **Đưa `n_estimators` vào GridSearchCV.** Lãng phí: thêm cây không bao giờ làm tệ đi. Đặt cố định 200 đến 500 và tune các tham số khác.
7. **Đọc `feature_importances_` như quan hệ nhân quả**, rồi quên rằng nó thiên vị feature nhiều giá trị. Luôn kiểm chứng chéo bằng `permutation_importance` trên tập test.
8. **Quên `stratify=y` khi split** với dữ liệu mất cân bằng: tập test có thể thiếu hẳn một lớp hiếm.
9. **Quên `class_weight='balanced'`** với dữ liệu mất cân bằng. Cả `DecisionTreeClassifier` lẫn `RandomForestClassifier` đều hỗ trợ (RF còn có `'balanced_subsample'`).
10. **Không cố định `random_state`.** Cây rất nhạy: chỉ đổi seed là số liệu trong báo cáo đã khác. Luôn cố định và ghi rõ.
11. **Kỳ vọng cây hồi quy ngoại suy được.** Ra ngoài miền train, `DecisionTreeRegressor` chỉ kéo ngang giá trị của lá ngoài cùng.
12. **Dùng OOB hoặc CV ngẫu nhiên cho dữ liệu chuỗi thời gian / dữ liệu nhóm.** Phải dùng `TimeSeriesSplit` hoặc `GroupKFold`, nếu không kết quả sẽ lạc quan giả tạo.


## Tổng kết

1. **Decision Tree**: dễ giải thích, nhưng dễ overfit nếu không giới hạn depth.
2. **Random Forest**: trung bình hoá nhiều cây → ổn định, ít overfit, gần như luôn tốt hơn DT đơn.
3. **OOB score** là validation "miễn phí" của RF (không cần chia thêm tập val).
4. Cả hai cho `feature_importance_`, công cụ tốt để hiểu dữ liệu.
5. **Tránh data leakage**: nếu dùng `Pipeline` với scaler, vẫn nhớ split TRƯỚC khi `fit`.

# BÀI TẬP VỀ NHÀ

## Bài 1: Gini vs Entropy
Train 2 Decision Tree với `criterion='gini'` và `criterion='entropy'`, các tham số khác giữ nguyên. So sánh test accuracy. Sự khác biệt có lớn không?

## Bài 2: GridSearchCV cho RF
Dùng `GridSearchCV` để tìm best hyperparams cho Random Forest:
- `n_estimators`: [50, 100, 200]
- `max_depth`: [3, 5, 7, None]
- `min_samples_leaf`: [1, 3, 5]

Báo cáo `best_params_` và best CV score. So sánh với RF mặc định.

*Gợi ý:* `from sklearn.model_selection import GridSearchCV; gs = GridSearchCV(rf, params, cv=5).fit(X_train, y_train)`.

## Bài 3: Tự cài Information Gain
Viết hàm `info_gain(y_parent, y_left, y_right)`:
1. Tính entropy của `y_parent`.
2. Tính trung bình có trọng số entropy của hai con.
3. Trả về phép trừ.

Test: với `y_parent = [0,0,0,1,1,1,1,1]`, `y_left = [0,0,0]`, `y_right = [1,1,1,1,1]` → IG phải bằng entropy của parent (vì hai con đều thuần) ≈ 0.954.

## Bài 4: Vẽ decision boundary 2D
Lấy 2 feature `Age` và `Na_to_K` của drug200. Train DT (max_depth=3) và RF (n_estimators=50, max_depth=3). Vẽ decision boundary 2D bằng `contourf`. So sánh: RF có boundary mượt hơn không?

## Bài 5: Feature importance của RF so với DT
Train cả DT (max_depth=5) và RF (n_estimators=100, max_depth=5). In `feature_importances_` của cả hai, vẽ bar chart so sánh. RF có ổn định hơn DT khi đổi `random_state` không? (chạy 5 lần với 5 seed khác nhau, đo std).

*Gợi ý:* lặp với `random_state in [0,1,2,3,4]`, lưu importance, so sánh `np.std`.